In [2]:
import re
import ast
import nltk
import random
import numpy as np
import pandas as pd

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
import torch.nn as nn
from sklearn.metrics import f1_score
from gensim.models import Word2Vec
from collections import Counter

In [3]:
RANDOM_STATE= 42
BATCH_SIZE = 128
LR = 2e-4
EPOCHS = 100

# rnn Config
HIDDEN_DIM=300
MAX_LEN=256

MAX_VOCAB=20000
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_ID = 0
UNK_ID = 1

# === 10 classes ===
NUM_CLASSES = 10
TARGET_CLASSES = ['earn','acq','money-fx','grain','crude',
                  'trade','interest','ship','wheat','corn']
label2id = {name: i for i, name in enumerate(TARGET_CLASSES)}
id2label = {i: name for name, i in label2id.items()}
print('label2id:', label2id)

label2id: {'earn': 0, 'acq': 1, 'money-fx': 2, 'grain': 3, 'crude': 4, 'trade': 5, 'interest': 6, 'ship': 7, 'wheat': 8, 'corn': 9}


# Load Reuters dataset (ModLewis_train.csv)
Load the CSV and filter to keep only the 10 target classes.

In [4]:
df = pd.read_csv('ModLewis_train.csv')
print('Original shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

Original shape: (13625, 13)
Columns: ['text', 'text_type', 'topics', 'lewis_split', 'cgis_split', 'old_id', 'new_id', 'places', 'people', 'orgs', 'exchanges', 'date', 'title']


,text,text_type,topics,lewis_split,cgis_split,old_id,new_id,places,people,orgs,exchanges,date,title
0,Showers continued throughout the week in\nthe ...,"""NORM""",['cocoa'],"""TRAIN""","""TRAINING-SET""","""5544""","""1""",['el-salvador' 'usa' 'uruguay'],[],[],[],26-FEB-1987 15:01:01.79,BAHIA COCOA REVIEW
1,Standard Oil Co and BP North America\nInc said...,"""NORM""",[],"""TRAIN""","""TRAINING-SET""","""5545""","""2""",['usa'],[],[],[],26-FEB-1987 15:02:20.00,STANDARD OIL &lt;SRD> TO FORM FINANCIAL UNIT
2,Texas Commerce Bancshares Inc's Texas\nCommerc...,"""NORM""",[],"""TRAIN""","""TRAINING-SET""","""5546""","""3""",['usa'],[],[],[],26-FEB-1987 15:03:27.51,TEXAS COMMERCE BANCSHARES &lt;TCB> FILES PLAN
3,BankAmerica Corp is not under\npressure to act...,"""NORM""",[],"""TRAIN""","""TRAINING-SET""","""5547""","""4""",['usa' 'brazil'],[],[],[],26-FEB-1987 15:07:13.72,TALKING POINT/BANKAMERICA &lt;BAC> EQUITY OFFER
4,The U.S. Agriculture Department\nreported the ...,"""NORM""",['grain' 'wheat' 'corn' 'barley' 'oat' 'sorghum'],"""TRAIN""","""TRAINING-SET""","""5548""","""5""",['usa'],[],[],[],26-FEB-1987 15:10:44.60,NATIONAL AVERAGE PRICES FOR FARMER-OWNED RESERVE


In [5]:
# ตรวจสอบว่า wheat กับ corn มีอยู่ใน dataset ไหม
import ast

df_check = pd.read_csv('ModLewis_train.csv')
all_topics = df_check['topics'].apply(ast.literal_eval)

# รวม topic ทั้งหมดที่มีใน dataset
from collections import Counter
topic_counter = Counter()
for topics_list in all_topics:
    topic_counter.update(topics_list)

# ดูว่า wheat กับ corn มีกี่แถว
print("wheat:", topic_counter.get('wheat', 0), "แถว")
print("corn:", topic_counter.get('corn', 0), "แถว")

# แสดง topic ทั้งหมดที่มีใน dataset (เรียงตามจำนวน)
print("\n=== ทุก topics ใน dataset ===")
for topic, count in topic_counter.most_common():
    marker = " ✅" if topic in ['earn','acq','money-fx','grain','crude','trade','interest','ship','wheat','corn'] else ""
    print(f"  {topic:20s}: {count:5d}{marker}")

wheat: 0 แถว
corn: 0 แถว

=== ทุก topics ใน dataset ===
  earn                :  2840 ✅
  acq                 :  1596 ✅
  crude               :   254 ✅
  trade               :   251 ✅
  money-fx            :   207 ✅
  interest            :   190 ✅
  money-supply        :   123
  money-fxinterest    :   109
  ship                :   108 ✅
  grainwheat          :   102
  sugar               :    97
  coffee              :    90
  gold                :    70
  graincorn           :    68
  money-fxdlr         :    65
  gnp                 :    58
  cpi                 :    54
  cocoa               :    46
  grain               :    41 ✅
  reserves            :    37
  jobs                :    37
  ipi                 :    33
  crudenat-gas        :    32
  copper              :    31
  alum                :    31
  rubber              :    31
  iron-steel          :    26
  nat-gas             :    24
  crudeship           :    24
  tradebop            :    23
  bop                 :    2

In [6]:
# แปลง topics จาก string "['earn']" เป็น list จริง ['earn']
df['topics'] = df['topics'].apply(ast.literal_eval)

# เลือก topic แรกที่ตรงกับ 10 classes ที่ต้องการ
df['label_name'] = df['topics'].apply(
    lambda x: next((t for t in x if t in TARGET_CLASSES), None)
)

# กรองเฉพาะแถวที่มี topic ตรง
df = df[df['label_name'].notna()].reset_index(drop=True)
df['label'] = df['label_name'].map(label2id)

print(f'จำนวนข้อมูลหลังกรอง: {len(df)}')
print(df['label_name'].value_counts())

จำนวนข้อมูลหลังกรอง: 5487
label_name
earn        2840
acq         1596
crude        254
trade        251
money-fx     207
interest     190
ship         108
grain         41
Name: count, dtype: int64


# Data preprocessing
- remove punctuation marks
- remove HTML tags
- remove URL's
- remove characters which are not letters or digits
- remove successive whitespaces
- convert the text to lower case
- strip whitespaces

In [7]:
idx = random.randint(0, len(df)-1)
before_process = df.iloc[idx]['text']

def process(x):
    x = re.sub(r'[,\.!?:()"\']', '', x)
    x = re.sub(r'<.*?>', ' ', x)
    x = re.sub(r'http\S+', ' ', x)
    x = re.sub(r'[^a-zA-Z0-9]', ' ', x)
    x = re.sub(r'\s+', ' ', x)
    return x.lower().strip()

df['text'] = df['text'].astype(str).apply(lambda x: process(x))
after_process = df.iloc[idx]['text']
print('Before:', before_process[:200])
print('After:', after_process[:200])

Before: US Sprint, the 50-50 telephone venture
of GTE Corp &lt;GTE> and United Telecommunications Inc &lt;UT> set up
last June, is optimistic despite expecting to report a net loss
of about 500 mln dlrs this 
After: us sprint the 50 50 telephone venture of gte corp lt gte and united telecommunications inc lt ut set up last june is optimistic despite expecting to report a net loss of about 500 mln dlrs this year d


Remove stopwords using nltk

In [8]:
sw_set = set(nltk.corpus.stopwords.words('english'))

def tokenize(text):
    return nltk.tokenize.word_tokenize(text)
    
def sw_remove(x):
    words = tokenize(x.lower())
    filtered_list = [word for word in words if word not in sw_set]
    return filtered_list

df['text'] = df['text'].apply(lambda x: sw_remove(x))
print('Example tokens:', df.iloc[idx]['text'][:20])

Example tokens: ['us', 'sprint', '50', '50', 'telephone', 'venture', 'gte', 'corp', 'lt', 'gte', 'united', 'telecommunications', 'inc', 'lt', 'ut', 'set', 'last', 'june', 'optimistic', 'despite']


# Data splitting and tokenization

In [9]:
from sklearn.model_selection import train_test_split

train_rev, tmp_rev, train_sent, tmp_sent = train_test_split(
    df['text'], df['label'], test_size=0.1, random_state=RANDOM_STATE)
test_rev, val_rev, test_sent, val_sent = train_test_split(
    tmp_rev, tmp_sent, test_size=0.5, random_state=RANDOM_STATE)

print('train:', train_rev.shape)
print('test:', test_rev.shape)
print('val:', val_rev.shape)

train: (4938,)
test: (274,)
val: (275,)


# Text to Vector by Word2Vec

In [10]:
import multiprocessing
print("CPU cores:", multiprocessing.cpu_count())
CPU_CORES = multiprocessing.cpu_count()

CPU cores: 20


In [11]:
tokenized_train = train_rev.tolist()
vectorize_model = Word2Vec(
    sentences=tokenized_train, 
    vector_size=HIDDEN_DIM,
    window=5,
    min_count=1,
    workers=CPU_CORES,
    sg=1
)

print(type(tokenized_train))
print(type(tokenized_train[0]))
print(tokenized_train[0][:10])

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


<class 'list'>
<class 'list'>
['nan']


In [12]:
X_train = train_rev 
X_val = val_rev
X_test = test_rev

y_train = (train_sent).astype(np.int64).to_numpy()
y_val   = (val_sent).astype(np.int64).to_numpy()
y_test  = (test_sent).astype(np.int64).to_numpy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("=========================")
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (4938,)
X_val: (275,)
X_test: (274,)
y_train: (4938,)
y_val: (275,)
y_test: (274,)


# สร้าง embedding matrix ตาม vocab for initial weight

In [13]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print('DEVICE:', DEVICE)

DEVICE: cuda


In [14]:
counter = Counter()
for t in train_rev:
    counter.update(t)

vocab = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID}
for i, (w, _) in enumerate(counter.most_common(MAX_VOCAB - 2), start=2):
    vocab[w] = i

def encode(tokens_list):
    ids = [vocab.get(t, vocab[UNK_TOKEN]) for t in tokens_list][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

vocab_size = len(vocab)
print('vocab_size:', vocab_size)

vocab_size: 20000


In [15]:
embedding_weight = np.zeros((vocab_size, HIDDEN_DIM), dtype=np.float32)

rng = np.random.default_rng(RANDOM_STATE)
embedding_weight[UNK_ID] = rng.normal(0, 0.01, size=(HIDDEN_DIM,)).astype(np.float32)

for word, idx in vocab.items():
    if word in (PAD_TOKEN, UNK_TOKEN):
        continue
    if word in vectorize_model.wv:
        embedding_weight[idx] = vectorize_model.wv[word]

print('embedding_weight shape:', embedding_weight.shape)

embedding_weight shape: (20000, 300)


# RNN Part

In [16]:
class ReutersDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.tolist()
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = encode(self.texts[idx])
        x = torch.tensor(ids, dtype=torch.long)
        y = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return x, y

In [17]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, output_dim=10, embedding_matrix=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(torch.tensor(embedding_matrix, dtype=torch.float32))
        self.embedding.weight.requires_grad = True
        
        self.rnn = nn.LSTM(embed_dim, embed_dim // 2, batch_first=True)
        self.fc1 = nn.Linear(embed_dim // 2, embed_dim // 4)
        self.relu = nn.ReLU()
        self.fc_dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(embed_dim // 4, output_dim)

    def forward(self, text):
        lengths = (text != 0).sum(dim=1)
        embedded = self.embedding(text)
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), enforce_sorted=False, batch_first=True)
        _, (hidden, _) = self.rnn(packed)
        out = hidden[-1, :, :]
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc_dropout(out)
        out = self.fc2(out)
        return out

In [18]:
model = RNN(vocab_size, embed_dim=HIDDEN_DIM, output_dim=NUM_CLASSES,
            embedding_matrix=embedding_weight).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()
print(model)

c:\Users\nawapol\anaconda3\envs\PT\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RNN(
  (embedding): Embedding(20000, 300, padding_idx=0)
  (rnn): LSTM(300, 150, batch_first=True)
  (fc1): Linear(in_features=150, out_features=75, bias=True)
  (relu): ReLU()
  (fc_dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=75, out_features=10, bias=True)
)


In [19]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total = 0.0, 0

    all_preds = []
    all_labels = []

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * y.size(0)
        total += y.size(0)

        preds = logits.argmax(dim=1)

        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(y.detach().cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)

    acc = (y_pred == y_true).mean()
    f1 = f1_score(y_true, y_pred, average='macro')
    
    return total_loss / total, acc, f1

In [20]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return running_loss / total, correct / total

In [21]:
def fit(model, train_loader, val_loader, optimizer, criterion, device, epochs):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc*100:.2f}% | "
            f"val_loss={val_loss:.4f} val_acc={val_acc*100:.2f}% val_f1={val_f1:.4f}"
        )

    return history

In [22]:
def test(model, test_loader, criterion, device):
    test_loss, test_acc, test_f1 = evaluate(model, test_loader, criterion, device)
    print(f"TEST | loss={test_loss:.4f} acc={test_acc*100:.2f}% f1={test_f1:.4f}")
    return test_loss, test_acc, test_f1

In [23]:
train_ds = ReutersDataset(X_train, y_train)
val_ds   = ReutersDataset(X_val, y_val)
test_ds  = ReutersDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Train and Test

In [24]:
history = fit(model, train_loader, val_loader, optimizer, criterion, DEVICE, epochs=EPOCHS)
test_loss, test_acc, test_f1 = test(model, test_loader, criterion, DEVICE)

Epoch 01 | train_loss=2.1057 train_acc=48.42% | val_loss=1.7532 val_acc=52.36% val_f1=0.0859
Epoch 02 | train_loss=1.4443 train_acc=53.71% | val_loss=1.2038 val_acc=67.27% val_f1=0.1821
Epoch 03 | train_loss=1.0635 train_acc=70.41% | val_loss=1.0107 val_acc=72.00% val_f1=0.1925
Epoch 04 | train_loss=0.9692 train_acc=70.01% | val_loss=0.8416 val_acc=72.00% val_f1=0.1933
Epoch 05 | train_loss=0.7618 train_acc=74.14% | val_loss=0.7737 val_acc=76.00% val_f1=0.2636
Epoch 06 | train_loss=0.7764 train_acc=74.18% | val_loss=0.8519 val_acc=69.09% val_f1=0.2428
Epoch 07 | train_loss=0.6613 train_acc=76.99% | val_loss=0.6425 val_acc=77.45% val_f1=0.2700
Epoch 08 | train_loss=0.6997 train_acc=75.64% | val_loss=0.6872 val_acc=74.55% val_f1=0.2567
Epoch 09 | train_loss=0.6162 train_acc=79.55% | val_loss=0.6354 val_acc=78.18% val_f1=0.2764
Epoch 10 | train_loss=0.5908 train_acc=79.42% | val_loss=0.6194 val_acc=77.45% val_f1=0.2732
Epoch 11 | train_loss=0.5725 train_acc=79.57% | val_loss=0.6567 val_ac

In [25]:
@torch.no_grad()
def predict_text(model, text, device):
    model.eval()

    # preprocess
    text = sw_remove(process(text))
    
    ids = encode(text)
    x = torch.tensor([ids], dtype=torch.long, device=device)

    logits = model(x)
    probs = torch.softmax(logits, dim=1).squeeze(0)

    pred_id = int(torch.argmax(probs).item())
    confidence = float(probs[pred_id].item()) * 100

    label = id2label[pred_id]
    return label, confidence, probs.detach().cpu().numpy()

In [26]:
sample_text = "Yugoslav government plans to stop subsidising loss-making firms will anger hundreds of thousands of workers, Western diplomats said. The law, proposed by Prime Minister Branko Mikulic, goes into effect on July 1 and aims to end a long-standing practice of supporting unprofitable companies. Under the law, wage cuts will be imposed on losing enterprises, while those failing to recover within a six-month grace period will face liquidation. The diplomats said Mikulic's attempt to create a market economy is inevitable, but has still come as a shock to those accustomed to government subsidies. It was a bitter pill which had to be swallowed, but if an overdose is taken too abruptly it may have adverse effects on the system, a Western diplomat said. He said if the law was applied too strictly it would probably provoke a new wave of strikes and unrest. Yugoslavia was swept by strikes last month following the introduction of a wage-freeze law, later amended to allow more flexibility and some exemptions in what some political analysts saw as a retreat by Mikulic. But with inflation moving towards 100 pct, trade union leaders have asked how much more deprivation workers can take. The union leaders said workers thoughout the country are already receiving salaries below limits set under existing law, while others have received no wages at all this year because their employers are unable to pay them. Workers also complain much of their income is taken in local, state and federal taxes. Many others are losing their motivation to work and confidence in government as they feel their decision-making powers are being eroded, trade union officials said. Meanwhile, the official Tanjug news agency reported a paper and cellulose factory at Ivangrad in the Montenegro republic closed yesterday and 2,000 of its workers were given temporary leave. Tanjug said the plant had been running at a loss for the 24 years it had been in operation, and its closure was the result of economic necessity rather than bankruptcy. REUTER"
label, conf, probs = predict_text(model, sample_text, DEVICE)
print(f"Predicted: {label} ({conf:.2f}%)")
print("\nAll class probabilities:")
for cls_name, prob in zip(TARGET_CLASSES, probs):
    print(f"  {cls_name:12s}: {prob*100:.2f}%")

Predicted: ship (53.76%)

All class probabilities:
  earn        : 9.89%
  acq         : 18.60%
  money-fx    : 2.72%
  grain       : 1.38%
  crude       : 10.69%
  trade       : 2.13%
  interest    : 0.82%
  ship        : 53.76%
  wheat       : 0.00%
  corn        : 0.00%
